In [21]:
import numpy as np
import pandas as pd
import os
import pathlib
from scipy.stats import gaussian_kde
from tqdm import tqdm
import warnings
from skimage import util
from skimage.filters import threshold_otsu, gaussian, threshold_isodata, threshold_minimum, threshold_triangle, threshold_otsu, threshold_mean, threshold_li, try_all_threshold, threshold_yen, threshold_multiotsu
from skimage.segmentation import clear_border, watershed, expand_labels
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, opening, ball, dilation, erosion
from skimage.transform import rescale
from skimage.feature import peak_local_max
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from scipy import stats as _stats
from matplotlib.colors import to_hex
from skimage.filters import median as median_filter
from skimage.morphology import square
from tifffile import imread
import tifffile
import napari
import colorsys
from collections import defaultdict
from skimage.morphology import closing, disk
import contextlib
from skimage.exposure import rescale_intensity

from PIL import ImageColor
from vtkmodules.vtkFiltersGeneral import vtkDiscreteFlyingEdges3D, vtkDiscreteMarchingCubes

import vtk
from vtk.util.numpy_support import numpy_to_vtk, vtk_to_numpy


In [18]:
# segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\segmented.tif")
# image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\Vimenting_SMA_VE-cadherin.lif - Ve-cad A488_SMA A594_zoom_3.tif")

segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\just_vasculature.tif")
image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\all_staining.tif")


fibroblasts = image_fluo[:,1,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)



C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\2301640185.py:21: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)


Kept label 1, area = 4675842.0 voxels


In [19]:
viewer = napari.Viewer()
# labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(5,2.2750,2.2750))
labels_layer = viewer.add_labels(binary_smoothed.astype(np.uint32), scale = (5,1.13,1.13))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
# viewer.add_image(fibroblasts, scale=(5,2.2750,2.2750))
viewer.add_image(fibroblasts, scale=(5, 1.13,1.13))

<Image layer 'fibroblasts' at 0x21d956c0450>

In [30]:
segmented = tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\segmentation.tif")
image_fluo= tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\all_channels.tif")

# Downsample by 4 in all spatial dims
DS = 4
segmented = segmented[::DS, ::DS, ::DS]
image_fluo = image_fluo[::DS, :, ::DS, ::DS]

cart = image_fluo[:,0,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)
print(f"Downsampled shape: {cart.shape}")

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\1737231175.py:22: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))


Kept label 1, area = 915543.0 voxels
Downsampled shape: (28, 719, 718)


In [37]:
# Add 20 slices of black padding on each side in Z
Z_PAD = 20
pad_seg = np.zeros((Z_PAD, *binary_smoothed.shape[1:]), dtype=binary_smoothed.dtype)
binary_smoothed = np.concatenate([pad_seg, binary_smoothed, pad_seg], axis=0)

pad_cart = np.zeros((Z_PAD, *cart.shape[1:]), dtype=cart.dtype)
cart = np.concatenate([pad_cart, cart, pad_cart], axis=0)

print(f"Padded shapes: cart={cart.shape}, segmentation={binary_smoothed.shape}")


Padded shapes: cart=(68, 719, 718), segmentation=(68, 719, 718)


In [38]:
# --- Ellipsoid at intensity-weighted COM of CART channel ---
# Adjust these radii (in voxels) by trial and error
radius_z = 30    # half-length along Z
radius_y = 90    # half-length along Y
radius_x = 90    # half-length along X

# Compute intensity-weighted centre of mass
com = ndi.center_of_mass(cart)
cz, cy, cx = com
cz -= 2
cx += 10
cy += 10
print(f"CART intensity COM: z={cz:.1f}, y={cy:.1f}, x={cx:.1f}")

# Build coordinate grids and evaluate ellipsoid equation
zz, yy, xx = np.ogrid[:cart.shape[0], :cart.shape[1], :cart.shape[2]]
ellipsoid_mask = (((zz - cz) / radius_z) ** 2 +
                  ((yy - cy) / radius_y) ** 2 +
                  ((xx - cx) / radius_x) ** 2) <= 1.0

cart_ellipsoid = ellipsoid_mask.astype(np.int32)
print(f"Ellipsoid voxels: {cart_ellipsoid.sum()}")


CART intensity COM: z=29.8, y=363.9, x=364.4
Ellipsoid voxels: 1017861


In [39]:
viewer = napari.Viewer()
labels_layer = viewer.add_labels(cart_ellipsoid.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'gray'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'darkorange'}
viewer.add_image(cart, scale=(2,1.1018,1.1018) )

<Image layer 'cart' at 0x21db949c750>

In [10]:
def tif_pixel_sizes(tif):
    """Return (z_um, y_um, x_um) from a tifffile TiffFile; any value is None if unavailable."""
    z_um = y_um = x_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    ij = tif.imagej_metadata or {}
    if "spacing" in ij:
        z_um = float(ij["spacing"])
    return z_um, y_um, x_um

In [27]:
image_path = r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_REPRESENTATIVE_IMAGE_OUTPUTS\Bel_mCh_BF_VASCUMAP_FL32_UTD4_Merged_6channel.tif"


In [28]:
test = imread(pathlib.Path(image_path))
with tifffile.TiffFile(image_path) as tif:
    z_um, y_um, x_um = tif_pixel_sizes(tif)
    print(f"Pixel sizes (z, y, x): {z_um}, {y_um}, {x_um}")
seg = test[:,-2,:,:].astype(np.uint8)
fluo = test[:,1,:,:]
bf = test[:,0,:,:]
# ------------------------------------------------------------
# NumPy -> vtkImageData
# ------------------------------------------------------------

z, y, x = seg.shape

vtk_image = vtk.vtkImageData()
vtk_image.SetDimensions(x, y, z)

# VTK coordinates are X, Y, Z
# Your napari scale is Z, Y, X = (10, 2, 2)
vtk_image.SetSpacing(x_um, y_um, z_um)

vtk_array = numpy_to_vtk(seg.ravel(order="C"), deep=True, array_type=vtk.VTK_UNSIGNED_CHAR,)

vtk_image.GetPointData().SetScalars(vtk_array)


surface_nets = vtk.vtkSurfaceNets3D()
surface_nets.SetInputData(vtk_image)

# Extract label 1
surface_nets.SetValue(0, 1)

# Built-in smoothing
surface_nets.SetSmoothing(True)
surface_nets.SetNumberOfIterations(500)

# Napari needs triangles
surface_nets.SetOutputMeshTypeToTriangles()

surface_nets.Update()

mesh = surface_nets.GetOutput()

# VTK vertices are X, Y, Z
vertices_xyz = vtk_to_numpy(mesh.GetPoints().GetData())

# Napari volume coordinates are Z, Y, X
vertices = vertices_xyz[:, [2, 1, 0]]

# vtkCellArray format:
# [3, i0, i1, i2,
#  3, i0, i1, i2,
#  ...]
polys = vtk_to_numpy(mesh.GetPolys().GetData())
faces = polys.reshape(-1, 4)[:, 1:]



Pixel sizes (z, y, x): 9.99954, 1.3000000000605358, 1.3000000000605358


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_345476\3261192547.py:53: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  polys = vtk_to_numpy(mesh.GetPolys().GetData())


In [29]:
fluo = rescale_intensity(fluo, out_range=(0, 255))
p_low, p_high = np.percentile(bf, (1, 99))
bf = rescale_intensity(bf, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)

In [ ]:
viewer = napari.Viewer()

viewer.add_surface((vertices, faces), name="smoothed segmentation", shading="smooth",)
viewer.add_image(fluo, scale=(z_um, y_um, x_um), name="fluorescence")
viewer.add_image(bf, scale=(z_um, y_um, x_um), name="brightfield")
viewer.add_labels(seg, scale=(z_um, y_um, x_um), name="segmentation")

<Labels layer 'segmentation' at 0x19d83082680>

Rendering frames...


  0%|          | 0/241 [00:02<?, ?it/s]


Rendering frames...


100%|██████████| 230/230 [07:56<00:00,  2.07s/it]


...

 23%|██▎       | 46/204 [00:21<01:15,  2.09it/s] 


Rendering frames...


 23%|██▎       | 46/202 [00:24<01:23,  1.86it/s] 


Rendering frames...


 23%|██▎       | 46/198 [00:21<01:09,  2.18it/s]
